# 00. Imports & Upload DB

In [ ]:
# Машинное обучение и метрики, реально используемые в ноутбуке
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from tqdm import tqdm

# Не используются в текущем AUTOWOE-пайплайне; оставлены как справка из прошлых экспериментов.
# from sklearn.metrics import (
#     accuracy_score,
#     precision_score,
#     recall_score,
#     f1_score,
#     confusion_matrix,
#     classification_report,
#     balanced_accuracy_score,
# )
# from catboost import CatBoostClassifier
# import torch
# from lightautoml.automl.presets.tabular_presets import TabularAutoML, TabularUtilizedAutoML
# from lightautoml.dataset.roles import DatetimeRole
# from lightautoml.tasks import Task
# from lightautoml.addons.tabular_interpretation import SSWARM
# from lightautoml.validation.np_iterators import TimeSeriesIterator
# from lightautoml.reader.guess_roles import gini_normalized


In [ ]:
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

# AutoWoE из LightAutoML
#from lightautoml.addons.autowoe import AutoWoE
from autowoe import AutoWoE, ReportDeco

### ---- Run flags


In [ ]:
# Флаги для безопасного Run All с чистого kernel.
# Основное обучение и базовые таблицы запускаются; тяжелые/экспортные блоки выключены по умолчанию.
RUN_SCORECARD_EXPORT = True
RUN_INTERPRETER_FEAT_EXPORT = True
RUN_GINI_TIME_TREND = True

# Тяжелые/интерактивные блоки: включать точечно, когда нужен соответствующий артефакт.
RUN_VIF_WOE_REPORT_3_8 = False
RUN_SPINE = False

# Внешние артефакты/отчеты: включать после финального пересчета модели.
RUN_AUTOWOE_HTML_REPORT = False
RUN_MODEL_EXPORT = False
RUN_SQL_EXPORT = False


### ---- Stability functions

In [ ]:
import os
import warnings
import logging

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

# Настройка логгирования
#logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

LOG_FILE = "stability_tags.log"
# Логирование stability-анализа не настраиваем при обычном Run All,
# чтобы не создавать файл stability_tags.log, пока функция явно не вызвана.
# При необходимости раскомментируйте блок ниже перед запуском analyze_feature_stability_optimized_prev.
# logging.basicConfig(
#     level=logging.INFO,
#     format="%(asctime)s - %(levelname)s - %(message)s",
#     handlers=[
#         logging.FileHandler(LOG_FILE, mode="a", encoding="utf-8"),
#         logging.StreamHandler(),
#     ],
#     force=True,
# )

def analyze_feature_stability_optimized_prev(df, feature_cols, date_col, save_plots = True):
    """
    Оптимизированный анализ стабильности признаков во времени.

    Параметры
    ----------
    df : pd.DataFrame
        Исходный датафрейм.
    feature_cols : list[str]
        Список колонок-признаков (например, TAG_*).
    date_col : str
        Название колонки с датой.

    Возвращает
    ----------
    dict
        {
            'fill_ratio_by_month': DataFrame - заполненность по месяцам,
            'mode_match_ratio_by_month': DataFrame - доля совпадений с модой по месяцам,
            'fill_stats': DataFrame - статистики по заполненности,
            'mode_stats': DataFrame - статистики по совпадению с модой,
            'stability_stats': DataFrame - объединённые статистики,
            'unstable_features': list[str] - список нестабильных признаков
        }
    """
    warnings.filterwarnings("ignore")
    logging.info("Начало анализа стабильности признаков")

    # --- 1. Подготовка данных -------------------------------------------------
    logging.info("Подготовка данных: приведение даты и фильтрация по дате, заполнение NaN нулями")
    df = df.copy()
    df[date_col] = pd.to_datetime(df[date_col])

    # Фильтр по дате (по аналогии с исходным кодом)
    #df = df[df[date_col] >= "2023-06-01"]
    df["year_month"] = df[date_col].dt.to_period("M")

    # 2. Заменяем все пропуски на 0
    df_filled = df.copy()
    df_filled[feature_cols] = df_filled[feature_cols].fillna(0)

    # 3. Группировка по месяцам
    gb = df_filled.groupby("year_month")

    # 4. Заполненность = доля != 0
    logging.info("Расчет заполненности по месяцам")
    fill_ratio_by_month = gb[feature_cols].apply(lambda x: (x != 0).mean())

    # --- 5. Глобальная мода для каждой фичи (игнорируем 0) -------------------
    logging.info("Поиск глобальной моды для каждой фичи (игнорируя 0)")
    df_nonzero = df[feature_cols].replace(0, pd.NA).dropna(how="all")
    if not df_nonzero.empty:
        global_mode = df_nonzero.mode(dropna=True).iloc[0]
    else:
        global_mode = pd.Series(dtype=object)

    # Если мода не найдена для какого-то признака — исключаем его из расчёта
    valid_modes = global_mode.dropna()
    logging.info(f"Найдены моды для {len(valid_modes)} фич из {len(feature_cols)}")

    # --- 6. Доля совпадений с модой по месяцам --------------------------------
    logging.info("Расчет доли совпадений с модой по месяцам")
    if len(valid_modes) == 0:
        mode_match_ratio_by_month = pd.DataFrame(
            index=fill_ratio_by_month.index,
            columns=feature_cols,
            dtype=float,
        )
    else:
        subset_cols = valid_modes.index.tolist()
        mode_vals = valid_modes.to_dict()

        match_df = df_filled[["year_month"] + subset_cols].copy()
        for col in tqdm(subset_cols, desc="Расчет совпадений с модой"):
            match_df[col] = (match_df[col] == mode_vals[col]).astype(int)

        mode_match_by_month = match_df.groupby("year_month")[subset_cols].mean()

        mode_match_ratio_by_month = pd.DataFrame(
            index=fill_ratio_by_month.index,
            columns=feature_cols,
            dtype=float,
        )
        mode_match_ratio_by_month[subset_cols] = mode_match_by_month.reindex(fill_ratio_by_month.index)

    # --- 7. Статистика по заполненности ---------------------------------------
    logging.info("Расчет статистики заполненности")
    fill_stats = pd.DataFrame({
        "min_fill": fill_ratio_by_month.min(),
        "max_fill": fill_ratio_by_month.max(),
        "var_fill": fill_ratio_by_month.var(),
    })

    # --- 8. Статистика по совпадениям с модой ---------------------------------
    logging.info("Расчет статистики доли совпадений с модой")
    mode_stats = pd.DataFrame({
        "min_mode_match": mode_match_ratio_by_month.min(),
        "max_mode_match": mode_match_ratio_by_month.max(),
        "var_mode": mode_match_ratio_by_month.var(),
    })

    # --- 9. Объединённая таблица ----------------------------------------------
    stability_stats = fill_stats.join(mode_stats, how="outer")

    # --- 10. Адаптивные пороги дисперсии (95-й перцентиль) --------------------
    var_fill_clean = fill_stats["var_fill"].dropna()
    var_mode_clean = mode_stats["var_mode"].dropna()

    var_threshold_fill = var_fill_clean.quantile(0.95) if not var_fill_clean.empty else 0.0
    var_threshold_mode = var_mode_clean.quantile(0.95) if not var_mode_clean.empty else 0.0

    logging.info(f"Порог дисперсии заполненности (95%): {var_threshold_fill:.4f}")
    logging.info(f"Порог дисперсии совпадений с модой (95%): {var_threshold_mode:.4f}")

    # --- 11. Относительные изменения между месяцами ---------------------------
    fill_changes_rel = fill_ratio_by_month.pct_change().abs()
    fill_changes_rel = fill_changes_rel.replace([np.inf, -np.inf], np.nan)

    mode_changes_rel = mode_match_ratio_by_month.pct_change().abs()
    mode_changes_rel = mode_changes_rel.replace([np.inf, -np.inf], np.nan)

    # --- 12. Квантили: 2.5% и 97.5% ------------------------------------------
    q25_fill = fill_ratio_by_month.quantile(0.025, axis=0)
    q975_fill = fill_ratio_by_month.quantile(0.975, axis=0)

    q25_mode = mode_match_ratio_by_month.quantile(0.025, axis=0)
    q975_mode = mode_match_ratio_by_month.quantile(0.975, axis=0)

    # --- 13. Относительная разница (max - min) / min --------------------------
    min_fill = fill_ratio_by_month.min()
    max_fill = fill_ratio_by_month.max()
    rel_diff_fill = np.where(
        min_fill > 0,
        (max_fill - min_fill) / min_fill,
        np.where(max_fill > 0, np.inf, 0),
    )
    rel_diff_fill = pd.Series(rel_diff_fill, index=min_fill.index)

    min_mode = mode_match_ratio_by_month.min()
    max_mode = mode_match_ratio_by_month.max()
    rel_diff_mode = np.where(
        min_mode > 0,
        (max_mode - min_mode) / min_mode,
        np.where(max_mode > 0, np.inf, 0),
    )
    rel_diff_mode = pd.Series(rel_diff_mode, index=min_mode.index)

    rel_diff_threshold = 10  # оставляю ваш порог, комментарий можно поправить при желании
    unstable_by_diff_fill = set(rel_diff_fill[rel_diff_fill > rel_diff_threshold].index)
    unstable_by_diff_mode = set(rel_diff_mode[rel_diff_mode > rel_diff_threshold * 2].index)

    # --- 14. Абсолютный критерий: средняя заполненность < 0.01 ----------------
    mean_fill = fill_ratio_by_month.mean()
    unstable_by_critical_low_fill = set(mean_fill[mean_fill < 0.001].index)

    # --- 15. Критерий: низкая заполненность до 2025 года и высокая в 2025 -----
    logging.info("Поиск признаков с низкой заполненностью до 2025 года и высокой в 2025 году")

    fill_before_2025 = fill_ratio_by_month[fill_ratio_by_month.index.year < 2025]
    fill_in_2025 = fill_ratio_by_month[fill_ratio_by_month.index.year == 2025]

    mean_fill_before_2025 = fill_before_2025.mean()
    mean_fill_in_2025 = fill_in_2025.mean()

    low_fill_before_2025 = mean_fill_before_2025[mean_fill_before_2025 < 0.1].index
    high_fill_in_2025 = mean_fill_in_2025[mean_fill_in_2025 >= 0.1].index

    # Признаки, которые были низко заполнены до 2025, но высоко заполнены в 2025
    stable_by_fill_change = set(low_fill_before_2025).intersection(high_fill_in_2025)

    # Исключаем их из списка нестабильных по критически низкой заполненности
    unstable_by_critical_low_fill = unstable_by_critical_low_fill - stable_by_fill_change

    # --- 16. Дисперсия --------------------------------------------------------
    unstable_by_var_fill = set(stability_stats[stability_stats["var_fill"] > var_threshold_fill].index)
    unstable_by_var_mode = set(stability_stats[stability_stats["var_mode"] > var_threshold_mode].index)

    # --- 17. Относительные изменения (более 3 месяцев) ------------------------
    n_rel_anom_fill = (fill_changes_rel > 0.99).sum()
    n_rel_anom_mode = (mode_changes_rel > 0.99).sum()
    unstable_by_rel_change_fill = set(n_rel_anom_fill[n_rel_anom_fill > 2].index)
    unstable_by_rel_change_mode = set(n_rel_anom_mode[n_rel_anom_mode > 2].index)

    # --- 18. Экстремумы (<2.5% или >97.5%, более 2 месяцев) -------------------
    n_low_fill = (fill_ratio_by_month < q25_fill).sum()
    n_high_fill = (fill_ratio_by_month > q975_fill).sum()
    n_low_mode = (mode_match_ratio_by_month < q25_mode).sum()
    n_high_mode = (mode_match_ratio_by_month > q975_mode).sum()

    unstable_by_extreme_fill_low = set(n_low_fill[n_low_fill > 2].index)
    unstable_by_extreme_fill_high = set(n_high_fill[n_high_fill > 2].index)
    unstable_by_extreme_mode_low = set(n_low_mode[n_low_mode > 2].index)
    unstable_by_extreme_mode_high = set(n_high_mode[n_high_mode > 2].index)

    # --- 19. Фильтр «признак отключён в 2025» ---------------------------------
    logging.info("Поиск признаков, отключённых в 2025 г. (>6 месяцев с 0 < fill < 0.001)")
    fill_2025 = fill_ratio_by_month[fill_ratio_by_month.index.year == 2025]
    n_near_zero_2025 = ((fill_2025 > 0) & (fill_2025 < 0.001)).sum()
    switched_off_2025 = set(n_near_zero_2025[n_near_zero_2025 > 6].index)

    # --- 20. Объединение всех критериев ---------------------------------------
    unstable_features = sorted(list(
        unstable_by_critical_low_fill |
        switched_off_2025 |
        unstable_by_var_fill |
        unstable_by_var_mode |
        unstable_by_rel_change_fill |
        unstable_by_rel_change_mode |
        unstable_by_extreme_fill_low |
        unstable_by_extreme_fill_high |
        unstable_by_extreme_mode_low |
        unstable_by_extreme_mode_high 
    ))

    stable_features = list(set(set(feature_cols) - set(unstable_features))) ### ADJUST

    # --- 21. Логгирование ------------------------------------------------------
    logging.info(f"Найдено {len(unstable_features)} нестабильных фич:")
    logging.info(f"  • fill < 0.001 (среднее): {len(unstable_by_critical_low_fill)}")
    logging.info(f"  • fill ~ 0 (отключены в 2025): {len(switched_off_2025)}")
    logging.info(f"  • var_fill > 95% перцентиля: {len(unstable_by_var_fill)}")
    logging.info(f"  • var_mode > 95% перцентиля: {len(unstable_by_var_mode)}")
    logging.info(f"  • rel_change_fill > 99% (более 3 мес): {len(unstable_by_rel_change_fill)}")
    logging.info(f"  • rel_change_mode > 99% (более 3 мес): {len(unstable_by_rel_change_mode)}")
    logging.info(f"  • extreme_fill_low (<2.5%, более 2 мес): {len(unstable_by_extreme_fill_low)}")
    logging.info(f"  • extreme_fill_high (>97.5%, более 2 мес): {len(unstable_by_extreme_fill_high)}")
    logging.info(f"  • extreme_mode_low (<2.5%, более 2 мес): {len(unstable_by_extreme_mode_low)}")
    logging.info(f"  • extreme_mode_high (>97.5%, более 2 мес): {len(unstable_by_extreme_mode_high)}")
    

    # --- 22. Построение и сохранение графиков ---------------------------------
    if save_plots:
        logging.info("Построение и сохранение графиков для нестабильных фич")

        output_dir = "FEATURES_STABILITY_UNSTABLE"
        os.makedirs(output_dir, exist_ok=True)

        x_dates = fill_ratio_by_month.index.to_timestamp()

        for feat in unstable_features:
            fig, ax = plt.subplots(figsize=(10, 5))

            ax.plot(x_dates, fill_ratio_by_month[feat].values,
                    label="Заполненность (≠0)", marker="o")
            ax.plot(x_dates, mode_match_ratio_by_month[feat].values,
                    label="Совпадение с модой", marker="x")

            ax.set_title(f"Фича: {feat}")
            ax.set_ylabel("Доля")
            ax.set_xlabel("Месяц")
            ax.legend()
            fig.autofmt_xdate()
            fig.tight_layout()

            # безопасное имя файла
            safe_name = "".join(
                c if c.isalnum() or c in ("_", "-") else "_" for c in str(feat)
            )
            save_path = os.path.join(output_dir, f"{safe_name}.png")
            fig.savefig(save_path, dpi=200)
            plt.close(fig)

            logging.info(f"Сохранён график: {save_path}")

# Построение стабильных
        if save_plots:
            logging.info("Построение и сохранение графиков для стабильных фич")

            output_dir = "FEATURES_STABILITY_STABLE"
            os.makedirs(output_dir, exist_ok=True)

            x_dates = fill_ratio_by_month.index.to_timestamp()

            for feat in stable_features:
                fig, ax = plt.subplots(figsize=(10, 5))

                ax.plot(x_dates, fill_ratio_by_month[feat].values,
                        label="Заполненность (≠0)", marker="o")
                ax.plot(x_dates, mode_match_ratio_by_month[feat].values,
                        label="Совпадение с модой", marker="x")

                ax.set_title(f"Фича: {feat}")
                ax.set_ylabel("Доля")
                ax.set_xlabel("Месяц")
                ax.legend()
                fig.autofmt_xdate()
                fig.tight_layout()

                # безопасное имя файла
                safe_name = "".join(
                    c if c.isalnum() or c in ("_", "-") else "_" for c in str(feat)
                )
                save_path = os.path.join(output_dir, f"{safe_name}.png")
                fig.savefig(save_path, dpi=200)
                plt.close(fig)

                logging.info(f"Сохранён график: {save_path}")

    logging.info("Анализ завершен")

    return {
        "fill_ratio_by_month": fill_ratio_by_month,
        "mode_match_ratio_by_month": mode_match_ratio_by_month,
        "fill_stats": fill_stats,
        "mode_stats": mode_stats,
        "stability_stats": stability_stats,
        "unstable_features": unstable_features,
        "stable_features": stable_features,
    }


### ---- columns type

In [ ]:
def get_columns_and_types(df_name, csv_prefix = None):
    columns_and_types = [(col, df_name[col].dtype.name) for col in df_name.columns]

    # Преобразуем этот список в DataFrame
    info_df = pd.DataFrame(columns_and_types, columns=['Column_name', 'Dtype'])
    if csv_prefix is not None:
        info_df.to_csv(f'{csv_prefix}_Columns_dtype_org.csv', sep=';', encoding = 'cp1251', decimal =',')

    # Группируем по типу данных и подсчитываем количество колонок каждого типа
    grouped = info_df.groupby('Dtype').size().reset_index(name='Columns_cnt')

    # Добавляем долю колонок для каждого типа данных
    total_columns = len(df_name.columns)
    grouped['Distr'] = grouped['Columns_cnt'] / total_columns * 100

    return grouped

### ---- HT NAMES CATALOG

In [ ]:
HT_catalog_path = r"E:\PubPricing\01_Retail\03_PPI\01_Modelling\01_HT\00_HT_Descr\pyCatalog\HT_names_prom_2026.csv"

if os.path.exists(HT_catalog_path):
    HT_catalog_csv = pd.read_csv(HT_catalog_path, low_memory=False, decimal='.', encoding='utf-8', sep=';')
    tags_name_map = HT_catalog_csv.set_index('NAME_LAMA')['TAG_NAME'].to_dict()
else:
    print(f'WARNING: HT catalog not found: {HT_catalog_path}. TAG_DESCRIPTION будет пустым.')
    HT_catalog_csv = pd.DataFrame(columns=['NAME_LAMA', 'TAG_NAME'])
    tags_name_map = {}

# Пример проверки расшифровки:
# print(tags_name_map.get("TAG_14232"))


### ---- FSA - OW gini and mixed-type correlation

In [ ]:
from scipy.stats import chi2_contingency

# ---------------------------
# ВСПОМОГАТЕЛЬНЫЕ ФУНКЦИИ
# ---------------------------

def _is_categorical(series: pd.Series, max_unique_ratio: float = 0.05, force_object_as_cat: bool = True) -> bool:
    """Эвристика: объектные/категориальные типы -> cat; числовые, но с очень малым числом уникальных значений -> cat."""
    if force_object_as_cat and (series.dtype == 'object' or str(series.dtype).startswith('category') or series.dtype == 'bool'):
        return True
    if np.issubdtype(series.dtype, np.number):
        nunique = series.nunique(dropna=True)
        return nunique <= max(10, int(len(series) * max_unique_ratio))
    return True

def _safe_numeric(s: pd.Series):
    """Преобразовать к float, заменить inf/−inf на NaN."""
    s = pd.to_numeric(s, errors='coerce')
    s = s.replace([np.inf, -np.inf], np.nan)
    return s

def _gini_normalized(y_true: np.ndarray, y_score: np.ndarray) -> float:
    """
    Нормализованный Gini по определению Kaggle:
    Gini(y, s) = 2*Lorenz(s) - 1, нормализуем на Gini(y, y).
    Здесь s — скор/ранг; для регрессии используем сам признак как скор.
    """
    # Уберём NaN синхронно
    mask = np.isfinite(y_true) & np.isfinite(y_score)
    y_true = np.asarray(y_true)[mask]
    y_score = np.asarray(y_score)[mask]
    if len(y_true) < 2:
        return np.nan

    def _gini(y, pred):
        # сортируем по pred по убыванию
        order = np.argsort(-pred, kind="mergesort")
        y = y[order]
        # накопленные суммы
        cum_y = np.cumsum(y)
        sum_y = cum_y[-1]
        if sum_y == 0:
            return 0.0
        # Лоренц-кривая на дискретной сетке
        lorenz = cum_y / sum_y
        n = len(y)
        g = (lorenz - (np.arange(1, n + 1) / n)).sum()
        return g / n

    g = _gini(y_true, y_score)
    g_perfect = _gini(y_true, y_true)  # нормирующая константа
    return np.nan if g_perfect == 0 else g / g_perfect

def _cramers_v(x: pd.Series, y: pd.Series) -> float:
    """Cramér’s V для двух категориальных признаков (с поправкой на размер таблицы)."""
    tbl = pd.crosstab(x, y)
    if tbl.size == 0:
        return np.nan
    chi2, _, _, _ = chi2_contingency(tbl, correction=False)
    n = tbl.values.sum()
    if n == 0:
        return np.nan
    phi2 = chi2 / n
    r, k = tbl.shape
    # корректировка Байкла–Крамера
    phi2corr = max(0, phi2 - (k - 1)*(r - 1)/(n - 1))
    rcorr = r - (r - 1)**2 / (n - 1)
    kcorr = k - (k - 1)**2 / (n - 1)
    denom = min((kcorr - 1), (rcorr - 1))
    return np.sqrt(phi2corr / denom) if denom > 0 else np.nan

def _correlation_ratio(categories: pd.Series, values: pd.Series) -> float:
    """Корреляционное отношение η(num ~ cat). categories — категориальная, values — числовая."""
    values = _safe_numeric(values)
    mask = categories.notna() & values.notna()
    if mask.sum() < 2:
        return np.nan
    categories = categories[mask]
    values = values[mask]

    groups = []
    for k, grp in values.groupby(categories):
        if len(grp) > 0:
            groups.append(grp.values)

    if len(groups) < 2:
        return np.nan
    all_vals = values.values
    y_mean = all_vals.mean()
    ss_between = sum([len(g) * (g.mean() - y_mean)**2 for g in groups])
    ss_total = ((all_vals - y_mean)**2).sum()
    return np.sqrt(ss_between / ss_total) if ss_total > 0 else np.nan

# ---------------------------
# 1) ОДНОФАКТОРНЫЙ GINI
# ---------------------------

def one_factor_gini_classification(df: pd.DataFrame, target: str) -> pd.DataFrame:
    """
    Для бинарной классификации: Gini = 2*AUC - 1,
    где в качестве score берём сам признак (для категориальных — среднее таргета по категориям).
    """
    y = df[target].values
    if set(np.unique(y)).issubset({0, 1}) is False:
        raise ValueError("Целевой признак должен быть бинарным (0/1).")

    out = []
    for col in tqdm(df.columns, desc="One-factor GINI (Classification)"):
        if col == target:
            continue
        s = df[col]
        # обработка по типу
        if _is_categorical(s):
            # target-mean encoding (simple, без кросс-валидации для one-factor оценки)
            means = df.groupby(col, dropna=False)[target].mean()
            score = s.map(means)
        else:
            score = _safe_numeric(s)

        # AUC может упасть, если все NaN или константа
        score = score.astype(float)
        # Импутация NaN медианой (иначе roc_auc_score упадёт)
        if score.notna().sum() == 0 or score.nunique(dropna=True) < 2:
            gini = np.nan
        else:
            filled = score.fillna(score.median())
            try:
                auc = roc_auc_score(y, filled)
                gini = 2 * auc - 1
            except ValueError:
                gini = np.nan

        out.append((col, gini))
    res = pd.DataFrame(out, columns=["feature", "gini"]).sort_values("gini", ascending=False, na_position="last")
    return res.reset_index(drop=True)

def one_factor_gini_regression(df: pd.DataFrame, target: str) -> pd.DataFrame:
    """
    Для регрессии: нормализованный Gini между y и «скором»,
    где скором является сам признак (для категориальных — среднее целевого по категориям).
    Это мера ранговой дискр.способности признака относительно y.
    """
    y = _safe_numeric(df[target]).values
    out = []
    for col in tqdm(df.columns, desc="One-factor GINI (Regression)"):
        if col == target:
            continue
        s = df[col]
        if _is_categorical(s):
            means = df.groupby(col, dropna=False)[target].mean()
            score = s.map(means)
        else:
            score = _safe_numeric(s)

        if pd.isna(score).all() or score.nunique(dropna=True) < 2:
            g = np.nan
        else:
            # Импутация медианой, чтобы удалять минимум наблюдений
            filled = score.fillna(score.median()).values
            g = _gini_normalized(y_true=y, y_score=filled)
        out.append((col, g))
    res = pd.DataFrame(out, columns=["feature", "gini_normalized"]).sort_values("gini_normalized", ascending=False, na_position="last")
    return res.reset_index(drop=True)

# ---------------------------
# 2) МАТРИЦА ПОПАРНЫХ КОРРЕЛЯЦИЙ ДЛЯ СМЕШАННЫХ ТИПОВ
# ---------------------------

def mixed_correlation_matrix(df: pd.DataFrame, prefer_spearman: bool = False) -> pd.DataFrame:
    """
    Возвращает квадратную матрицу «смешанной корреляции»:
    - num–num: Pearson (или Spearman, если prefer_spearman=True)
    - cat–cat: Cramér’s V
    - num–cat: η (correlation ratio)
    """
    cols = df.columns.tolist()
    types = {c: ('cat' if _is_categorical(df[c]) else 'num') for c in cols}
    n = len(cols)
    mat = np.zeros((n, n), dtype=float)
    mat[:] = np.nan

    # Предрасчёт числовых и категориальных представлений
    num_data = {c: _safe_numeric(df[c]) for c in cols}
    cat_data = {c: df[c].astype('object') for c in cols}

    # Быстрый блок для num-num (симметрично)
    num_cols = [c for c in cols if types[c] == 'num']
    if prefer_spearman and len(num_cols) >= 2:
        # Спирмен
        ranks = pd.DataFrame({c: num_data[c].rank(method='average') for c in num_cols})
        pear = ranks.corr(method='pearson', min_periods=2)
        for i, ci in enumerate(num_cols):
            for j, cj in enumerate(num_cols):
                mat[cols.index(ci), cols.index(cj)] = pear.iloc[i, j]
    elif len(num_cols) >= 2:
        pear = pd.DataFrame({c: num_data[c] for c in num_cols}).corr(method='pearson', min_periods=2)
        for i, ci in enumerate(num_cols):
            for j, cj in enumerate(num_cols):
                mat[cols.index(ci), cols.index(cj)] = pear.iloc[i, j]
    elif len(num_cols) == 1:
        mat[cols.index(num_cols[0]), cols.index(num_cols[0])] = 1.0

    # Остальные пары
    for i, ci in tqdm(enumerate(cols), total=len(cols), desc="Mixed Correlation Matrix"):
        for j, cj in enumerate(cols):
            if i == j:
                mat[i, j] = 1.0
                continue
            if not np.isnan(mat[i, j]):
                continue  # уже заполнено блоком num-num
            ti, tj = types[ci], types[cj]
            if ti == 'cat' and tj == 'cat':
                r = _cramers_v(cat_data[ci], cat_data[cj])
            elif ti == 'num' and tj == 'cat':
                r = _correlation_ratio(cat_data[cj], num_data[ci])
            elif ti == 'cat' and tj == 'num':
                r = _correlation_ratio(cat_data[ci], num_data[cj])
            else:
                # этот случай закрыт выше, но на всякий
                r = pd.concat([num_data[ci], num_data[cj]], axis=1).corr(method='pearson', min_periods=2).iloc[0, 1]
            mat[i, j] = r

    return pd.DataFrame(mat, index=cols, columns=cols)

In [ ]:
# EG of use
if False:
    print("\n=== One-factor GINI (classification) ===")
    print(one_factor_gini_classification(df, target="y_bin"))

    print("\n=== One-factor normalized GINI (regression) ===")
    print(one_factor_gini_regression(df, target="y_reg"))

    print("\n=== Mixed-type correlation matrix (Pearson for num–num) ===")
    corr = mixed_correlation_matrix(df.drop(columns=["y_bin","y_reg"]), prefer_spearman=False)
    print(corr.round(3))

### ---- GINI Time Trend

#### ------ Import и функции расчета GINI и ДИ

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from scipy.stats import norm
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings("ignore")


def gini_binary(y_true, y_pred):
    """Считает Gini через AUC."""
    auc = roc_auc_score(y_true, y_pred)
    return 2 * auc - 1


def gini_customized(y_true, y_pred, gini_normalized_bool=False):
    """Backward-compatible wrapper для старых вызовов в ноутбуке."""
    if gini_normalized_bool:
        raise ValueError("Новая логика Gini Time Trend поддерживает только бинарную классификацию")
    return gini_binary(y_true, y_pred)


def compute_confidence_interval(gini, n, alpha=0.05):
    """Простой нормальный аппрокс. ДИ."""
    se = np.sqrt((gini * (1 - gini)) / n) if n > 0 else 0
    z = norm.ppf(1 - alpha / 2)
    return gini - z * se, gini + z * se


#### ------ расчет динамики GINI: возвращает pd DateFrame

In [ ]:
def calc_gini_dynamics_binary(df1, observed_col, predicted_col, date_col,
                              freq='M', min_observations=100,
                              alpha_95=0.05, alpha_99=0.01):
    """
    Расчет динамики Gini (бинарная классификация) по календарным периодам.
    Периоды хранятся как pandas.Period, чтобы график не сдвигал месяцы/кварталы.
    """

    freq = str(freq).upper()

    data = df1.copy()
    data[date_col] = pd.to_datetime(data[date_col])

    freq_map = {
        'D': 'D',
        'W': 'W',
        'M': 'M',
        'Q': 'Q',
        'Y': 'Y'
    }

    period_freq = freq_map.get(freq, 'M')
    grouped = data.groupby(data[date_col].dt.to_period(period_freq), sort=True)

    results = []

    for period, group_data in grouped:
        if len(group_data) >= min_observations:
            try:
                gini_model = gini_binary(group_data[observed_col], group_data[predicted_col])
                n_obs = len(group_data)

                ci_95_low, ci_95_high = compute_confidence_interval(gini_model, n_obs, alpha_95)
                ci_99_low, ci_99_high = compute_confidence_interval(gini_model, n_obs, alpha_99)

                actual = group_data[observed_col].mean()
                fitted = group_data[predicted_col].mean()
                diff_to_pred = actual / fitted if fitted != 0 else np.nan

                results.append({
                    "period": period,
                    "gini": gini_model,
                    "n_observations": n_obs,
                    "ci_95_low": ci_95_low,
                    "ci_95_high": ci_95_high,
                    "ci_99_low": ci_99_low,
                    "ci_99_high": ci_99_high,
                    "value_actual": actual,
                    "value_fitted": fitted,
                    "value_parity_diff_to_pred": diff_to_pred,
                })
            except Exception as e:
                print(f"Ошибка в периоде {period}: {e}")
                continue

    if not results:
        print("Нет данных для графика")
        return None

    return pd.DataFrame(results).sort_values('period').reset_index(drop=True)


def calc_gini_dynamics(df1, observed_col, predicted_col, date_col,
                       freq='M', min_observations=100,
                       alpha_95=0.05, alpha_99=0.01,
                       gini_normalized_bool=False):
    """Backward-compatible wrapper для старых вызовов в ноутбуке."""
    if gini_normalized_bool:
        raise ValueError("Новая логика Gini Time Trend поддерживает только бинарную классификацию")

    return calc_gini_dynamics_binary(
        df1=df1,
        observed_col=observed_col,
        predicted_col=predicted_col,
        date_col=date_col,
        freq=freq,
        min_observations=min_observations,
        alpha_95=alpha_95,
        alpha_99=alpha_99,
    )


#### ------ построение графика динамики GINI

In [ ]:
def plot_gini_dynamics_binary(df1, observed_col, predicted_col, date_col,
                              freq='M', min_observations=100,
                              alpha_95=0.05, alpha_99=0.01,
                              figsize=(14, 8), title=None):
    """
    Динамика Gini (бинарная классификация) с доверительными интервалами.
    """

    freq = str(freq).upper()

    data = df1.copy()
    data[date_col] = pd.to_datetime(data[date_col])

    results_df1 = calc_gini_dynamics_binary(
        data,
        observed_col,
        predicted_col,
        date_col,
        freq=freq,
        min_observations=min_observations,
        alpha_95=alpha_95,
        alpha_99=alpha_99,
    )

    if results_df1 is None:
        return None, None

    x_positions = np.arange(len(results_df1))

    fig, ax1 = plt.subplots(figsize=figsize)
    ax2 = ax1.twinx()

    # Столбцы = число наблюдений
    ax2.bar(x_positions, results_df1['n_observations'],
            alpha=0.3, color="#2ca02c", label="Количество наблюдений", width=0.6)

    # 99% ДИ
    ax1.fill_between(x_positions, results_df1['ci_99_low'], results_df1['ci_99_high'],
                     alpha=0.3, color="#ff4444", label="99% ДИ")

    # 95% ДИ
    ax1.fill_between(x_positions, results_df1['ci_95_low'], results_df1['ci_95_high'],
                     alpha=0.5, color="#ffcc00", label="95% ДИ")

    # Линия Gini
    ax1.plot(x_positions, results_df1['gini'], color='black',
             linewidth=2, marker='o', markersize=6, label="Gini")

    # подписи на точках
    for x_pos, (_, row) in zip(x_positions, results_df1.iterrows()):
        ax1.annotate(f"{row['gini']:.3f}",
                     (x_pos, row['gini']),
                     textcoords="offset points", xytext=(0, 10),
                     ha='center', fontsize=9, fontweight='bold')

    # Пороговые линии
    ax1.axhline(y=gini_binary(data[observed_col], data[predicted_col]),
                color='purple', linestyle='--', alpha=0.7, label='Среднее Gini')
    ax1.axhline(y=0.2, color='red', linestyle='--', alpha=0.7, label='Порог 20%')
    ax1.axhline(y=0.3, color='green', linestyle='--', alpha=0.7, label='Порог 30%')

    # Оси
    ax1.set_xlabel("Период", fontsize=12, fontweight='bold')
    ax1.set_ylabel("Gini", fontsize=12, fontweight='bold', color="#1f77b4")
    ax2.set_ylabel("Количество наблюдений", fontsize=12, fontweight='bold', color="#2ca02c")

    ax1.tick_params(axis='y', labelcolor="#1f77b4")
    ax2.tick_params(axis='y', labelcolor="#2ca02c")

    # Формат периодов без визуального сдвига к следующему месяцу/кварталу
    def _format_period(period):
        if freq == 'Q':
            return f"{period.year}-Q{period.quarter}"
        if freq == 'Y':
            return f"{period.year}"
        if freq == 'M':
            return period.start_time.strftime('%Y-%m')
        if freq == 'W':
            return f"{period.start_time:%Y-%m-%d}\n{period.end_time:%Y-%m-%d}"
        return period.start_time.strftime('%Y-%m-%d')

    ax1.set_xticks(x_positions)
    ax1.set_xticklabels([_format_period(period) for period in results_df1['period']])
    ax1.set_xlim(-0.5, len(results_df1) - 0.5)

    plt.setp(ax1.xaxis.get_majorticklabels(), rotation=45, ha='right')

    if title is None:
        title = "Динамика Gini (бинарная классификация)"
    ax1.set_title(title, fontsize=14, fontweight='bold', pad=20)

    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2,
               loc='upper left', frameon=True, fancybox=True, shadow=True)

    ax1.grid(True, alpha=0.3)
    plt.tight_layout()

    print(f"Построен график для {len(results_df1)} периодов")
    print(f"Gini: {gini_binary(data[observed_col], data[predicted_col]):.3f}")
    print(f"Мин Gini: {results_df1['gini'].min():.3f}")
    print(f"Макс Gini: {results_df1['gini'].max():.3f}")
    print(f"Среднее кол-во наблюдений: {results_df1['n_observations'].mean():.0f}")

    return fig, ax1


def plot_gini_dynamics(df1, observed_col, predicted_col, date_col,
                       freq='M', min_observations=100,
                       alpha_95=0.05, alpha_99=0.01,
                       figsize=(14, 8), title=None,
                       gini_normalized_bool=False):
    """Backward-compatible wrapper для старых вызовов в ноутбуке."""
    if gini_normalized_bool:
        raise ValueError("Новая логика Gini Time Trend поддерживает только бинарную классификацию")

    return plot_gini_dynamics_binary(
        df1=df1,
        observed_col=observed_col,
        predicted_col=predicted_col,
        date_col=date_col,
        freq=freq,
        min_observations=min_observations,
        alpha_95=alpha_95,
        alpha_99=alpha_99,
        figsize=figsize,
        title=title,
    )


### --- Upload DB

In [ ]:
main_db_path = r"E:\PubPricing\01_Retail\03_PPI\01_Modelling\01_HT\2026m03\00_DB\radar_property_w_tags_ness_fields_prom_v5_2026_01_not_mort_pt_23_25.csv"
if not os.path.exists(main_db_path):
    raise FileNotFoundError(f'Не найден основной датасет: {main_db_path}')

HH_main_df = pd.read_csv(
    main_db_path,
    low_memory=False,
    sep='^',
    encoding='cp1251',
    decimal=','
)
HH_main_df.set_index('CO_KEY', inplace=True)

print(HH_main_df.shape)
HH_main_df.head()


### --- Target & Time setup

In [ ]:
TARGET_NAME       = 'ACC_CNT_WATER_BOOL'
TIME_TREND_FACTOR = 'POLICY_END_DATE'

HH_main_df[TIME_TREND_FACTOR] = pd.to_datetime(HH_main_df[TIME_TREND_FACTOR])

print('Target rate:', HH_main_df[TARGET_NAME].mean().round(4))
print('Date range: ', HH_main_df[TIME_TREND_FACTOR].min(), '→', HH_main_df[TIME_TREND_FACTOR].max())

### --- Train / Test split

In [ ]:
RANDOM_STATE = 28
TEST_SIZE    = 0.4

df_train, df_test = train_test_split(
    HH_main_df,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=HH_main_df[TARGET_NAME]
)

print(f'Train: {df_train.shape},  Test: {df_test.shape}')
print(f'Target rate — Train: {df_train[TARGET_NAME].mean():.4f},  Test: {df_test[TARGET_NAME].mean():.4f}')

# 01. Feature short list

In [ ]:
# Загружаем short list признаков (аналогично оригинальному ноутбуку)
feat_short_list_path = r'FSA_FeatTruncList02.csv'
if not os.path.exists(feat_short_list_path):
    raise FileNotFoundError(f'Не найден файл short list признаков: {feat_short_list_path}')

FeatShortList = (
    pd.read_csv(feat_short_list_path, low_memory=False, sep=';', encoding='cp1251', decimal=',')['NAME']
    .dropna()
    .drop_duplicates()
    .tolist()
)

# Дополнительные продуктовые признаки
feat_list_internal = [
    'PRODUCT_OBJECT_TYPE_HOUSE',
    'PRODUCT_OBJECT_TYPE_FLAT',
    'PRODUCT_OBJECT_TYPE_OTHER',
]
FeatShortList = list(dict.fromkeys(FeatShortList + feat_list_internal))

# Оставляем только те, что реально есть в данных
FeatShortList = [f for f in FeatShortList if f in df_train.columns]
print(f'Признаков в short list: {len(FeatShortList)}')


# 02. AutoWoE — обучение модели

### Параметры AutoWoE

| Параметр | Описание |
|---|---|
| `monotonic` | Монотонность биннинга (True = монотонный WoE) |
| `max_bin_count` | Макс. число бинов на признак |
| `pearson_th` | Порог корреляции Пирсона для отбора (дропаем коррелирующие) |
| `metric_th` | Минимальный AUC/Gini-like metric признака для включения в модель |
| `vif_th` | Порог VIF (Variance Inflation Factor) |
| `imp_th` | Порог важности признака |
| `regularized_refit` | Регуляризованное переобучение финальной LR |

### --- Типы признаков

In [ ]:
# AutoWoE требует явного указания типов: 'real' или 'cat'
# По умолчанию числовые -> 'real', object -> 'cat'

features_type = {}
for col in FeatShortList:
    if df_train[col].dtype == object or df_train[col].nunique() <= 10:
        features_type[col] = 'cat'
    else:
        features_type[col] = 'real'

print('real:', sum(v == 'real' for v in features_type.values()),
      '| cat:', sum(v == 'cat' for v in features_type.values()))

### --- Инициализация и обучение

In [ ]:
autowoe = AutoWoE(
    monotonic=False,        # без принудительной монотонности
    max_bin_count=5,        # не более 5 бинов на признак
    select_type=None,       # отбор на основе AUC/IV
    pearson_th=0.9,         # дропаем если корреляция > 0.9
    metric_th=0.505,        # минимальный AUC/Gini-like metric для включения
    vif_th=10.0,            # VIF порог
    imp_th=0,               # порог важности
    target_name=TARGET_NAME,
    regularized_refit=False,
    p_val=0.05,             # p-value для отбора в LR
    verbose=2,
)
autowoe = ReportDeco(autowoe, )

In [ ]:
%%time
autowoe.fit(
    df_train[FeatShortList + [TARGET_NAME]],
    target_name=TARGET_NAME,
    features_type=features_type,
)

# 03. Predict & Метрики

### --- Предсказание

In [ ]:
train_prediction = autowoe.model.predict_proba(df_train)

In [ ]:
test_prediction = autowoe.predict_proba(df_test)

In [ ]:
print("GINI Train:\t", 2*roc_auc_score(df_train[TARGET_NAME].values, train_prediction)-1)
print("GINI Test:\t", 2*roc_auc_score(df_test[TARGET_NAME].values, test_prediction)-1)

In [ ]:
df_train['Pred_fin_LAMA'] = train_prediction

In [ ]:
df_test['Pred_fin_LAMA'] = test_prediction

In [ ]:
if False:
    df_train[FeatShortList + [TIME_TREND_FACTOR, TARGET_NAME, 'Pred_fin_LAMA']].to_csv("df_train_scored.csv", sep=';', encoding = 'cp1251', decimal =',')
    df_test[FeatShortList + [TIME_TREND_FACTOR, TARGET_NAME, 'Pred_fin_LAMA']].to_csv("df_test_scored.csv", sep=';', encoding = 'cp1251', decimal =',')

### --- Features list

In [ ]:
model_represenation_dict = autowoe.get_model_represenation()

In [ ]:
# Selected feats
selected_feats_binary = sorted(list(model_represenation_dict.get('features')))
print('LightAutoML AUTOWOE has selected {} feats!'.format(len(selected_feats_binary)))
for i in selected_feats_binary:
    print(i)

### --- Scorecard


In [ ]:
from collections import defaultdict


def get_tag_description(variable, tags_name_map):
    """Возвращает расшифровку TAG_* из каталога HT_names_prom_2026.csv."""
    if pd.isna(variable):
        return None

    variable = str(variable).strip()
    if variable == "Intercept":
        return None

    # На случай технических суффиксов AutoWoE/LightAutoML.
    tag_name = variable.split("__F__")[0].strip()
    return tags_name_map.get(tag_name)


def make_autowoe_scorecard(autowoe_object, tags_name_map=None):
    """
    Собирает scorecard AutoWoE в формате отчета:
    Variable / TAG_DESCRIPTION / Value / WOE / COEF / POINTS.
    """
    tags_name_map = tags_name_map or {}
    model = autowoe_object.model if hasattr(autowoe_object, "model") else autowoe_object

    def round_ext(x):
        # Округление как в AutoWoE report: -0.0 превращаем в 0.0.
        return 0 + round(float(x), 2)

    rows = []
    intercept = round_ext(model.intercept)
    rows.append({
        "Variable": "Intercept",
        "TAG_DESCRIPTION": None,
        "Value": None,
        "WOE": None,
        "COEF": intercept,
        "POINTS": intercept,
    })

    for feature, coef in model.features_fit.items():
        woe = model.woe_dict[feature]
        coef = round_ext(coef)

        if woe.f_type == "cat":
            cat_split = defaultdict(list)
            special_values = {key for key in woe.cod_dict if isinstance(key, str)}
            for category_value, bin_id in woe.split.items():
                if category_value not in special_values:
                    cat_split[bin_id].append(category_value)

        for key, w in woe.cod_dict.items():
            if isinstance(key, str):
                label = str(key)
            elif woe.f_type == "cat":
                label = ", ".join(str(x) for x in cat_split.get(key, []))
            elif key == 0:
                if len(woe.split) == 0:
                    label = f"-inf < {feature} < +inf"
                else:
                    label = f"{feature} <= {round_ext(woe.split[int(key)])}"
            elif key == len(woe.split):
                label = f"{feature} > {round_ext(woe.split[int(key - 1)])}"
            else:
                label = f"{round_ext(woe.split[int(key - 1)])} < {feature} <= {round_ext(woe.split[int(key)])}"

            woe_value = round_ext(w)
            rows.append({
                "Variable": feature,
                "TAG_DESCRIPTION": get_tag_description(feature, tags_name_map),
                "Value": label,
                "WOE": woe_value,
                "COEF": coef,
                "POINTS": round_ext(woe_value * coef),
            })

    return pd.DataFrame(rows, columns=["Variable", "TAG_DESCRIPTION", "Value", "WOE", "COEF", "POINTS"])


if RUN_SCORECARD_EXPORT:
    Scorecard_tbl = make_autowoe_scorecard(autowoe, tags_name_map)
    Scorecard_tbl.to_csv('Scorecard_tbl.csv', sep=';', encoding='cp1251', decimal=',', index=False, na_rep='None')

    print(f'Scorecard rows: {len(Scorecard_tbl)}')
    print('Scorecard saved: Scorecard_tbl.csv')
    display(Scorecard_tbl.fillna('None'))
else:
    print('Scorecard export skipped: RUN_SCORECARD_EXPORT=False')


### --- Interpreter feat / Report 2.1


In [ ]:
def make_interpreter_feat_tbl(autowoe_object, tags_name_map=None):
    """
    Inline-таблица раздела 2.1 AutoWoE report:
    признак + регрессионный коэффициент, дополненная расшифровкой TAG_*.
    """
    tags_name_map = tags_name_map or {}
    model = autowoe_object.model if hasattr(autowoe_object, "model") else autowoe_object

    rows = []
    features_fit = model.features_fit.sort_values()
    for feature, coef in features_fit.items():
        rows.append({
            "INTERPRETER_FEAT": str(feature),
            "TAG_DESCRIPTION": get_tag_description(feature, tags_name_map),
            "REGRESSION_COEF": 0 + round(float(coef), 6),
            "ABS_REGRESSION_COEF": 0 + round(abs(float(coef)), 6),
        })

    return pd.DataFrame(
        rows,
        columns=["INTERPRETER_FEAT", "TAG_DESCRIPTION", "REGRESSION_COEF", "ABS_REGRESSION_COEF"],
    )


if RUN_INTERPRETER_FEAT_EXPORT:
    InterpreterFeat_tbl = make_interpreter_feat_tbl(autowoe, tags_name_map)
    InterpreterFeat_tbl.to_csv(
        'InterpreterFeat_tbl.csv',
        sep=';',
        encoding='cp1251',
        decimal=',',
        index=False,
        na_rep='None',
    )

    print(f'Interpreter feats: {len(InterpreterFeat_tbl)}')
    print('Interpreter feat saved: InterpreterFeat_tbl.csv')
    display(InterpreterFeat_tbl.fillna('None'))
else:
    print('Interpreter feat export skipped: RUN_INTERPRETER_FEAT_EXPORT=False')


### --- VIF & WOE split by significant factors / Report 3.8


In [ ]:
try:
    from IPython.display import display, Markdown
except ImportError:
    def display(obj):
        print(obj)

    class Markdown(str):
        pass

import os


def calc_vif_like_autowoe_report(data_enc):
    """VIF как в разделе 3.8 AutoWoE report: diag(inv(corr(X)))."""
    if data_enc.shape[1] < 2:
        return []

    cc = np.corrcoef(data_enc.values, rowvar=False)
    if cc.ndim < 2:
        return []

    try:
        vif_values = np.linalg.inv(cc).diagonal()
    except np.linalg.LinAlgError:
        # Защита от сингулярной корреляционной матрицы; порядок и смысл VIF сохраняем максимально близко.
        vif_values = np.linalg.pinv(cc).diagonal()

    vif_values = np.round(vif_values, 6)
    return sorted(zip(data_enc.columns, vif_values), key=lambda item: item[1], reverse=True)


def make_vif_interpreter_tbl(autowoe_object, train_df, tags_name_map=None):
    """
    Таблица значимых факторов по убыванию VIF:
    фактор, расшифровка, VIF, regression coefficient.
    """
    tags_name_map = tags_name_map or {}
    model = autowoe_object.model if hasattr(autowoe_object, "model") else autowoe_object

    significant_feats = list(model.features_fit.index)
    train_enc = model.test_encoding(train_df, feats=significant_feats)
    vif_rows = calc_vif_like_autowoe_report(train_enc)

    coef_map = model.features_fit.to_dict()
    rows = []
    for feature, vif in vif_rows:
        rows.append({
            "FACTOR": str(feature),
            "TAG_DESCRIPTION": get_tag_description(feature, tags_name_map),
            "VIF": 0 + round(float(vif), 6),
            "REGRESSION_COEF": 0 + round(float(coef_map.get(feature, np.nan)), 6),
            "ABS_REGRESSION_COEF": 0 + round(abs(float(coef_map.get(feature, np.nan))), 6),
        })

    return pd.DataFrame(
        rows,
        columns=["FACTOR", "TAG_DESCRIPTION", "VIF", "REGRESSION_COEF", "ABS_REGRESSION_COEF"],
    )


def plot_woe_split_inline(autowoe_object, feature, output_dir="VIF_WOE_PLOTS"):
    """Строит WOE-разбивку как в разделе 2.2 отчета и сохраняет PNG."""
    os.makedirs(output_dir, exist_ok=True)
    model = autowoe_object.model if hasattr(autowoe_object, "model") else autowoe_object
    woe_series = model.get_woe(feature)

    fig, ax = plt.subplots(figsize=(15, 5))
    ax.bar(woe_series.index.astype(str), woe_series.values, color="g")
    ax.set_title(f"Split of feature {feature} and woe values")
    ax.set_xlabel("Bins", fontsize=14)
    ax.set_ylabel("WoE values", fontsize=14)
    ax.tick_params(axis="x", labelrotation=20)
    ax.grid(True, alpha=0.3)
    fig.tight_layout()

    safe_name = "".join(c if c.isalnum() or c in ("_", "-") else "_" for c in str(feature))
    save_path = os.path.join(output_dir, f"{safe_name}_woe.png")
    fig.savefig(save_path, dpi=200, bbox_inches="tight")
    return fig, ax, save_path


if RUN_VIF_WOE_REPORT_3_8:
    VIFInterpreter_tbl = make_vif_interpreter_tbl(autowoe, df_train, tags_name_map)
    VIFInterpreter_tbl.to_csv(
        'VIFInterpreter_tbl.csv',
        sep=';',
        encoding='cp1251',
        decimal=',',
        index=False,
        na_rep='None',
    )

    print(f'VIF significant factors: {len(VIFInterpreter_tbl)}')
    print('VIF interpreter table saved: VIFInterpreter_tbl.csv')
    display(VIFInterpreter_tbl.fillna('None'))

    for _, row in VIFInterpreter_tbl.iterrows():
        factor = row["FACTOR"]
        tag_description = tags_name_map.get(str(factor).split("__F__")[0].strip())

        print("=" * 120)
        print(f"FACTOR: {factor}")
        print(f"TAG_DESCRIPTION: {tag_description}")
        print(f"VIF: {row['VIF']}")
        print(f"Regression coefficient: {row['REGRESSION_COEF']}")

        display(Markdown(
            f"**{factor}**  \n"
            f"TAG_DESCRIPTION: `{tag_description}`  \n"
            f"VIF: `{row['VIF']}`  \n"
            f"Regression coefficient: `{row['REGRESSION_COEF']}`"
        ))
        fig, ax, save_path = plot_woe_split_inline(autowoe, factor)
        display(fig)
        plt.close(fig)
        print(f"WOE plot saved: {save_path}")
else:
    print('VIF/WOE report 3.8 skipped: RUN_VIF_WOE_REPORT_3_8=False')


### --- SPINE: individual WOE waterfall


In [ ]:
try:
    from IPython.display import display, Markdown
except ImportError:
    def display(obj):
        print(obj)

    class Markdown(str):
        pass

import os


SPINE_OBSERVATION_ID = None  # можно указать CO_KEY / index конкретного наблюдения
SPINE_OBSERVATION_POS = 0    # используется, если SPINE_OBSERVATION_ID = None
SPINE_TOP_N = 30             # сколько факторов показывать на графике; остаток агрегируется в OTHER_FACTORS


def sigmoid(x):
    return 1 / (1 + np.exp(-x))


def get_spine_observation(df, observation_id=None, observation_pos=0):
    """Возвращает одну строку для построения SPINE."""
    if observation_id is not None:
        if observation_id not in df.index:
            raise KeyError(f"Observation id {observation_id!r} not found in dataframe index")
        return df.loc[[observation_id]].copy(), observation_id

    observation_key = df.index[observation_pos]
    return df.iloc[[observation_pos]].copy(), observation_key


def format_spine_value(value):
    if pd.isna(value):
        return "nan"
    if isinstance(value, (float, np.floating)):
        return f"{float(value):.6f}"
    return str(value)


def get_feature_raw_value(row_df, feature):
    feature = str(feature)
    base_feature = feature.split("__F__")[0]

    if feature in row_df.columns:
        return row_df[feature].iloc[0]
    if base_feature in row_df.columns:
        return row_df[base_feature].iloc[0]
    return np.nan


def get_woe_bin_label(model, feature, woe_value):
    """Подбирает label бина по WOE значению из model.get_woe(feature)."""
    try:
        woe_split = model.get_woe(feature)
        diffs = (woe_split.astype(float) - float(woe_value)).abs()
        return str(diffs.idxmin())
    except Exception:
        return None


def make_spine_tbl(autowoe_object, df, observation_id=None, observation_pos=0,
                   tags_name_map=None, sort_by_abs=True):
    """
    SPINE для одного наблюдения:
    intercept + сумма линейных WOE-вкладов feature_coef * feature_woe.
    """
    tags_name_map = tags_name_map or {}
    model = autowoe_object.model if hasattr(autowoe_object, "model") else autowoe_object

    row_df, observation_key = get_spine_observation(
        df,
        observation_id=observation_id,
        observation_pos=observation_pos,
    )

    features = list(model.features_fit.index)
    woe_encoded = model.test_encoding(row_df, feats=features).iloc[0]
    coef_map = model.features_fit.to_dict()

    vif_map = {}
    if "VIFInterpreter_tbl" in globals():
        vif_map = VIFInterpreter_tbl.set_index("FACTOR")["VIF"].to_dict()

    rows = []
    for feature in features:
        raw_value = get_feature_raw_value(row_df, feature)
        woe_value = float(woe_encoded[feature])
        coef = float(coef_map[feature])
        contribution = woe_value * coef

        rows.append({
            "OBSERVATION_ID": observation_key,
            "FACTOR": str(feature),
            "TAG_DESCRIPTION": get_tag_description(feature, tags_name_map),
            "RAW_VALUE": raw_value,
            "RAW_VALUE_PRINT": format_spine_value(raw_value),
            "WOE_BIN": get_woe_bin_label(model, feature, woe_value),
            "WOE": 0 + round(woe_value, 6),
            "REGRESSION_COEF": 0 + round(coef, 6),
            "LINEAR_CONTRIBUTION": 0 + round(contribution, 6),
            "ABS_LINEAR_CONTRIBUTION": 0 + round(abs(contribution), 6),
            "VIF": vif_map.get(str(feature), np.nan),
        })

    spine_tbl = pd.DataFrame(rows)
    if sort_by_abs:
        spine_tbl = spine_tbl.sort_values("ABS_LINEAR_CONTRIBUTION", ascending=False).reset_index(drop=True)

    intercept = float(model.intercept)
    cumulative_logit = intercept
    cumulative_values = []
    for contribution in spine_tbl["LINEAR_CONTRIBUTION"]:
        cumulative_logit += float(contribution)
        cumulative_values.append(cumulative_logit)

    spine_tbl["CUMULATIVE_LOGIT"] = np.round(cumulative_values, 6)
    spine_tbl["CUMULATIVE_PROB"] = np.round(sigmoid(spine_tbl["CUMULATIVE_LOGIT"].astype(float)), 8)

    final_logit = intercept + spine_tbl["LINEAR_CONTRIBUTION"].astype(float).sum()
    final_prob = sigmoid(final_logit)
    try:
        model_prob = float(model.predict_proba(row_df)[0])
    except Exception:
        model_prob = np.nan

    spine_summary = pd.DataFrame([{
        "OBSERVATION_ID": observation_key,
        "INTERCEPT_LOGIT": 0 + round(intercept, 6),
        "FINAL_LOGIT": 0 + round(final_logit, 6),
        "FINAL_PROB": 0 + round(final_prob, 8),
        "MODEL_PREDICT_PROBA": 0 + round(model_prob, 8) if pd.notna(model_prob) else np.nan,
        "PROBA_DIFF": 0 + round(final_prob - model_prob, 10) if pd.notna(model_prob) else np.nan,
        "N_FACTORS": len(features),
    }])

    return spine_tbl, spine_summary, row_df


def plot_spine_waterfall(spine_tbl, spine_summary, top_n=30, output_dir="SPINE_PLOTS"):
    """Водопад линейных WOE-вкладов от intercept до финального logit/probability."""
    os.makedirs(output_dir, exist_ok=True)

    intercept = float(spine_summary["INTERCEPT_LOGIT"].iloc[0])
    final_logit = float(spine_summary["FINAL_LOGIT"].iloc[0])
    final_prob = float(spine_summary["FINAL_PROB"].iloc[0])
    observation_id = spine_summary["OBSERVATION_ID"].iloc[0]

    plot_tbl = spine_tbl.copy()
    if top_n is not None and len(plot_tbl) > top_n:
        top_tbl = plot_tbl.iloc[:top_n].copy()
        other_tbl = plot_tbl.iloc[top_n:].copy()
        other_contribution = other_tbl["LINEAR_CONTRIBUTION"].astype(float).sum()
        other_row = {
            "FACTOR": "OTHER_FACTORS",
            "TAG_DESCRIPTION": f"Остальные факторов: {len(other_tbl)}",
            "RAW_VALUE_PRINT": "",
            "WOE_BIN": "",
            "WOE": np.nan,
            "REGRESSION_COEF": np.nan,
            "LINEAR_CONTRIBUTION": other_contribution,
            "ABS_LINEAR_CONTRIBUTION": abs(other_contribution),
            "VIF": np.nan,
        }
        plot_tbl = pd.concat([top_tbl, pd.DataFrame([other_row])], ignore_index=True)

    cumulative_before = []
    cumulative_after = []
    current = intercept
    for contribution in plot_tbl["LINEAR_CONTRIBUTION"].astype(float):
        cumulative_before.append(current)
        current += contribution
        cumulative_after.append(current)

    plot_tbl["_before"] = cumulative_before
    plot_tbl["_after"] = cumulative_after

    height = max(7, min(18, 0.45 * len(plot_tbl) + 3))
    fig, ax = plt.subplots(figsize=(14, height))

    y_pos = np.arange(len(plot_tbl))
    colors = np.where(plot_tbl["LINEAR_CONTRIBUTION"].astype(float) >= 0, "#f06b75", "#2da44e")

    for y, (_, row), color in zip(y_pos, plot_tbl.iterrows(), colors):
        before = float(row["_before"])
        after = float(row["_after"])
        contribution = float(row["LINEAR_CONTRIBUTION"])
        left = min(before, after)
        width = abs(contribution)
        ax.barh(y, width, left=left, color=color, edgecolor="white", height=0.72)

        text_x = after + (0.015 if contribution >= 0 else -0.015)
        ha = "left" if contribution >= 0 else "right"
        ax.text(text_x, y, f"{contribution:+.6f}", va="center", ha=ha, fontsize=9, color=color)

    y_labels = [
        f"{row['RAW_VALUE_PRINT']} = {row['FACTOR']}"
        if row["FACTOR"] != "OTHER_FACTORS"
        else "OTHER_FACTORS"
        for _, row in plot_tbl.iterrows()
    ]
    ax.set_yticks(y_pos)
    ax.set_yticklabels(y_labels, fontsize=9)
    ax.invert_yaxis()

    ax.axvline(intercept, color="#888888", linestyle="--", linewidth=1)
    ax.axvline(final_logit, color="#222222", linestyle="-", linewidth=1)
    ax.text(intercept, -0.9, f"intercept = {intercept:.6f}", ha="center", va="bottom", fontsize=10, color="#666666")
    ax.text(final_logit, len(plot_tbl) - 0.2, f"logit = {final_logit:.6f}\nprob = {final_prob:.8f}",
            ha="center", va="top", fontsize=10, color="#222222")

    ax.set_xlabel("Linear score / logit = intercept + sum(WOE * coef)")
    ax.set_title(f"SPINE WOE waterfall for observation {observation_id}")
    ax.grid(axis="x", alpha=0.25)
    fig.tight_layout()

    safe_id = "".join(c if c.isalnum() or c in ("_", "-") else "_" for c in str(observation_id))
    save_path = os.path.join(output_dir, f"SPINE_{safe_id}.png")
    fig.savefig(save_path, dpi=200, bbox_inches="tight")
    return fig, ax, save_path


if RUN_SPINE:
    Spine_tbl, Spine_summary_tbl, Spine_observation_df = make_spine_tbl(
        autowoe,
        df_test,
        observation_id=SPINE_OBSERVATION_ID,
        observation_pos=SPINE_OBSERVATION_POS,
        tags_name_map=tags_name_map,
        sort_by_abs=True,
    )

    Spine_tbl.to_csv('Spine_tbl.csv', sep=';', encoding='cp1251', decimal=',', index=False, na_rep='None')
    Spine_summary_tbl.to_csv('Spine_summary_tbl.csv', sep=';', encoding='cp1251', decimal=',', index=False, na_rep='None')

    print(f"SPINE observation: {Spine_summary_tbl['OBSERVATION_ID'].iloc[0]}")
    print('SPINE table saved: Spine_tbl.csv')
    print('SPINE summary saved: Spine_summary_tbl.csv')
    display(Spine_summary_tbl.fillna('None'))
    display(Spine_tbl.fillna('None'))

    fig, ax, spine_plot_path = plot_spine_waterfall(Spine_tbl, Spine_summary_tbl, top_n=SPINE_TOP_N)
    display(fig)
    plt.close(fig)
    print(f"SPINE plot saved: {spine_plot_path}")
else:
    print('SPINE skipped: RUN_SPINE=False')


In [ ]:
SummaryStats_tbl = pd.DataFrame({
    'MODE': ['AUTOWOE'],
    'FEATURE_GROUP_SIZE': ['NA'],
    'MAX_TUNING_TIME_M': ['NA'],
    'METRIC': ['NA'],
    'CV_FOLDS': ['NA'],
    'TEST_SIZE': [TEST_SIZE],
    'USED_FEATS': len(sorted(list(model_represenation_dict.get('features')))),
    'TRAIN_GINI': [2*roc_auc_score(df_train[TARGET_NAME].values, train_prediction)-1],
    'OOF_GINI': ['NA'],
    'TEST_GINI': [2*roc_auc_score(df_test[TARGET_NAME].values, test_prediction)-1],
    'MODEL_STRUCTURE' : ['AUTOWOE']
})

In [ ]:
SummaryStats_tbl

In [ ]:
SummaryStats_tbl.to_csv('SummaryStats_tbl.csv', sep=';', encoding = 'cp1251', decimal =',')

#### --- Make GINI Time Trend

#### -------- GINI TRAIN

In [ ]:
if RUN_GINI_TIME_TREND:
    oot_df1 = pd.DataFrame(df_train[TARGET_NAME])
    oot_df1['FIN_pred'] = pd.DataFrame(df_train['Pred_fin_LAMA'])
    oot_df1['Date'] = df_train[TIME_TREND_FACTOR]
else:
    oot_df1 = None
    print('GINI time trend skipped: RUN_GINI_TIME_TREND=False')


In [ ]:
if RUN_GINI_TIME_TREND:
    plot_gini_dynamics(oot_df1, TARGET_NAME, "FIN_pred", "Date", freq="M", title="GINI TREND TRAIN", gini_normalized_bool=False)


##### --------- save to csv

In [ ]:
if RUN_GINI_TIME_TREND:
    gini_time_trend_train = calc_gini_dynamics(oot_df1, TARGET_NAME, "FIN_pred", "Date", freq="M", gini_normalized_bool=False)
    gini_time_trend_train.to_csv('gini_time_trend_train.csv', sep=';', encoding='cp1251', decimal=',')
else:
    gini_time_trend_train = None


In [ ]:
if RUN_GINI_TIME_TREND:
    gini_time_trend_train_q = calc_gini_dynamics(oot_df1, TARGET_NAME, "FIN_pred", "Date", freq="Q", gini_normalized_bool=False)
    gini_time_trend_train_q.to_csv('gini_time_trend_train_q.csv', sep=';', encoding='cp1251', decimal=',')
else:
    gini_time_trend_train_q = None


#### --------➡️ GINI TEST

In [ ]:
if RUN_GINI_TIME_TREND:
    oot_df2 = pd.DataFrame(df_test[TARGET_NAME])
    oot_df2['FIN_pred'] = pd.DataFrame(df_test['Pred_fin_LAMA'])
    oot_df2['Date'] = df_test[TIME_TREND_FACTOR]

    plot_gini_dynamics(oot_df2, TARGET_NAME, "FIN_pred", "Date", freq="M", title="GINI TREND TEST", gini_normalized_bool=False)

    gini_time_trend_test = calc_gini_dynamics(oot_df2, TARGET_NAME, "FIN_pred", "Date", freq="M", gini_normalized_bool=False)
    gini_time_trend_test.to_csv('gini_time_trend_test.csv', sep=';', encoding='cp1251', decimal=',')

    gini_time_trend_test_q = calc_gini_dynamics(oot_df2, TARGET_NAME, "FIN_pred", "Date", freq="Q", gini_normalized_bool=False)
    gini_time_trend_test_q.to_csv('gini_time_trend_test_q.csv', sep=';', encoding='cp1251', decimal=',')
else:
    oot_df2 = None
    gini_time_trend_test = None
    gini_time_trend_test_q = None


In [ ]:
if RUN_GINI_TIME_TREND:
    display(gini_time_trend_test)


In [ ]:
if RUN_GINI_TIME_TREND:
    display(gini_time_trend_test_q)


# 04. Интерпретация модели - Generate report

In [ ]:
report_params = {"output_path": "HT_HH_PROB_WD_IFL_model_report", # folder for report generation
                 "report_name": "SBS #TAGS MODELLING NONE-MOTOR. WHITEBOX REPORT",
                 "report_version_id": 1,
                  "city": "Moscow",
                  "model_aim": "Probability of water damage in Household Insurance products",
                  "model_name": "HH HT WD PROB",
                  "zakazchik": "SBS",
                  "high_level_department": "SBS",
                  "ds_name": "SBS",
                  "target_descr": "water damage",
                  "non_target_descr": "no claims appear"
}

if RUN_AUTOWOE_HTML_REPORT:
    autowoe.generate_report(report_params, )
else:
    print('AutoWoE HTML report skipped: RUN_AUTOWOE_HTML_REPORT=False')

# 05. Сохранение модели и результатов

### --- Сохранение модели

In [ ]:
if RUN_MODEL_EXPORT:
    joblib.dump(autowoe.model, 'HT_HH_PROB_WD_AUTOWOE_model.pkl')
    print('Модель сохранена: HT_HH_PROB_WD_AUTOWOE_model.pkl')
else:
    print('Model export skipped: RUN_MODEL_EXPORT=False')


### --- SQL

In [ ]:
sql_query = None
if RUN_SQL_EXPORT:
    sql_query = autowoe.model.get_sql_inference_query('global_temp.TABLE_1')
else:
    print('SQL export skipped: RUN_SQL_EXPORT=False')


In [ ]:
if RUN_SQL_EXPORT and sql_query is not None:
    print(sql_query)


In [ ]:
if RUN_SQL_EXPORT and sql_query is not None:
    print(type(sql_query))


In [ ]:
if RUN_SQL_EXPORT and sql_query is not None:
    with open("sql_query.txt", "w", encoding="utf-8") as file:
        file.write(sql_query)
